# PDF Parsing & Chunking

## Overview

This document explains how two functions form the first half of the ingestion pipeline: `extract_pages()` takes a PDF file and returns text organized by page, and `chunk_pages()` splits that text into overlapping segments suitable for embedding.

These two functions are the entry point for every document in the system. When a user uploads a PDF, the ingestion service runs it through extract → chunk → embed → store. This document covers the first two steps.

## Architecture Context

In the ingestion service (`services/ingestion/app/`), these live in `pdf_parser.py` and `chunker.py`. The `/ingest` endpoint calls `extract_pages()` first, then passes the result to `chunk_pages()`, then sends chunks to the embedder.

In [1]:
# Prerequisites — no external services needed
# pip install pypdf2 langchain-text-splitters fpdf2
from PyPDF2 import PdfReader, PdfWriter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from fpdf import FPDF
print("All packages loaded!")

All packages loaded!


## Package Introductions

### PyPDF2
PyPDF2 is a pure-Python library for reading and manipulating PDF files. We chose it because:
- **No system dependencies** — unlike `pdfminer` (complex) or `poppler` (requires C library install), PyPDF2 is pure Python and installs cleanly everywhere.
- **Simple API** — create a `PdfReader`, iterate `.pages`, call `.extract_text()` on each page.
- **Good enough for text-heavy PDFs** — it struggles with complex layouts, scanned images, or tables, but for documents with flowing text (reports, papers, manuals) it works well.

Note: PyPDF2 is technically deprecated in favor of `pypdf` (same authors, renamed). We're using PyPDF2 here because the API is identical and it's what the app uses. In a new project, use `pypdf`.

### LangChain Text Splitters
LangChain is a large framework for building LLM applications. We're only using one small piece: `langchain-text-splitters`. This sub-package provides text chunking strategies.

`RecursiveCharacterTextSplitter` is the one we use. It tries to split text at natural boundaries (paragraphs first, then sentences, then words) while keeping chunks under a target size. The "recursive" part means it tries multiple separators in order: `\n\n`, `\n`, ` `, then individual characters.

### fpdf2
A simple library for creating PDFs programmatically. We only use this to create test PDFs in the notebook — it's not part of the actual app.

## Go/TS Comparison

| Concept | Go | Python |
|---------|-----|--------|
| In-memory file buffer | `bytes.Buffer` / `io.Reader` | `io.BytesIO` |
| Reading a file-like object | `io.ReadAll(reader)` | `file.read()` / `file.seek(0)` |
| Error for bad input | `return nil, fmt.Errorf(...)` | `raise ValueError(...)` |

The `BytesIO` pattern is important — in Python, many libraries expect a "file-like object" (something with `.read()` and `.seek()` methods). `BytesIO` wraps raw bytes to look like a file. This is exactly like wrapping `[]byte` in a `bytes.Reader` in Go.

## Build It

### Step 1: Create a test PDF

First, let's create a PDF with known content so we can verify our extraction works.

In [2]:
from fpdf import FPDF
from io import BytesIO

def create_test_pdf() -> bytes:
    """Create a 2-page PDF with known content."""
    pdf = FPDF()

    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.cell(0, 10, "Page 1: The quarterly revenue was 2.5 million dollars.")
    pdf.ln()
    pdf.cell(0, 10, "Operating margins improved to 23 percent.")
    pdf.ln()
    pdf.cell(0, 10, "The company expanded into three new markets.")

    pdf.add_page()
    pdf.cell(0, 10, "Page 2: The engineering team grew to 45 people.")
    pdf.ln()
    pdf.cell(0, 10, "Customer satisfaction scores reached 4.8 out of 5.")
    pdf.ln()
    pdf.cell(0, 10, "Infrastructure costs decreased by 12 percent.")

    buffer = BytesIO()
    pdf.output(buffer)
    return buffer.getvalue()

pdf_bytes = create_test_pdf()
print(f"Created test PDF: {len(pdf_bytes)} bytes")

Created test PDF: 1588 bytes


### Step 2: Extract text from each page

This function is the equivalent of `pdf_parser.py` in the ingestion service. It takes a file-like object (BytesIO) and returns a list of dicts, one per page.

Note the error handling pattern: we check for empty input first, then catch any PyPDF2 parsing errors and wrap them in a `ValueError`. This is similar to how you'd return `fmt.Errorf("invalid PDF: %w", err)` in Go.

In [3]:
from io import BytesIO
from PyPDF2 import PdfReader

def extract_pages(pdf_file: BytesIO) -> list[dict]:
    """Extract text from each page of a PDF.

    Returns a list of dicts with 'page_number' (1-indexed) and 'text' keys.
    Raises ValueError if the file is empty or not a valid PDF.
    """
    try:
        content = pdf_file.read()
        if not content:
            raise ValueError("empty or invalid PDF")
        pdf_file.seek(0)
        reader = PdfReader(pdf_file)
    except Exception as e:
        if "empty or invalid" in str(e):
            raise
        raise ValueError(f"empty or invalid PDF: {e}")

    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"page_number": i + 1, "text": text})

    return pages

# Test it
pages = extract_pages(BytesIO(pdf_bytes))
for page in pages:
    print(f"\n--- Page {page['page_number']} ---")
    print(page["text"][:200])


--- Page 1 ---
Page 1: The quarterly revenue was 2.5 million dollars.
Operating margins improved to 23 percent.
The company expanded into three new markets.

--- Page 2 ---
Page 2: The engineering team grew to 45 people.
Customer satisfaction scores reached 4.8 out of 5.
Infrastructure costs decreased by 12 percent.


In [4]:
# Test error handling
import traceback

# Empty input
try:
    extract_pages(BytesIO(b""))
except ValueError as e:
    print(f"Empty input: {e}")

# Invalid PDF
try:
    extract_pages(BytesIO(b"not a pdf at all"))
except ValueError as e:
    print(f"Invalid PDF: {e}")

Empty input: empty or invalid PDF
Invalid PDF: empty or invalid PDF: EOF marker not found


### Step 3: Chunk the extracted text

Now we need to split the page text into smaller pieces for embedding. Why?

1. **LLM context windows** — embedding models have a maximum input length. Long pages need to be split.
2. **Retrieval precision** — smaller chunks mean more precise search results. A 100-word chunk about revenue is more useful than a 5000-word page that mentions revenue once.
3. **Overlap** — chunks overlap so that context isn't lost at boundaries. If a sentence spans two chunks, the overlap ensures both chunks contain the full sentence.

`RecursiveCharacterTextSplitter` handles all of this. You set `chunk_size` (target characters per chunk) and `chunk_overlap` (how many characters adjacent chunks share).

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pages(
    pages: list[dict],
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> list[dict]:
    """Split page text into overlapping chunks.

    Returns list of dicts with 'text', 'page_number', and 'chunk_index' keys.
    Empty pages are skipped.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    chunks = []
    index = 0
    for page in pages:
        text = page["text"].strip()
        if not text:
            continue

        splits = splitter.split_text(text)
        for split in splits:
            chunks.append({
                "text": split,
                "page_number": page["page_number"],
                "chunk_index": index,
            })
            index += 1

    return chunks

# Test with our PDF — use small chunk size to see splitting in action
chunks = chunk_pages(pages, chunk_size=100, chunk_overlap=20)
print(f"Pages: {len(pages)}, Chunks: {len(chunks)}\n")

for chunk in chunks:
    print(f"Chunk {chunk['chunk_index']} (page {chunk['page_number']}): {chunk['text'][:80]}...")

Pages: 2, Chunks: 4

Chunk 0 (page 1): Page 1: The quarterly revenue was 2.5 million dollars.
Operating margins improve...
Chunk 1 (page 1): The company expanded into three new markets....
Chunk 2 (page 2): Page 2: The engineering team grew to 45 people.
Customer satisfaction scores rea...
Chunk 3 (page 2): Infrastructure costs decreased by 12 percent....


> **Pitfall:** Claude Code sometimes imports from `langchain.text_splitter` (old, deprecated path) instead of `langchain_text_splitters` (the current standalone package). The old import may work if you have the full `langchain` package installed, but it pulls in hundreds of unnecessary dependencies. Always use `langchain-text-splitters` as a standalone package.

## Experiment

In [6]:
# Experiment 1: How does chunk_size affect the number of chunks?
for size in [50, 100, 200, 500, 1000]:
    chunks = chunk_pages(pages, chunk_size=size, chunk_overlap=0)
    print(f"chunk_size={size:>4}, overlap=0   → {len(chunks)} chunks")

chunk_size=  50, overlap=0   → 8 chunks
chunk_size= 100, overlap=0   → 4 chunks
chunk_size= 200, overlap=0   → 2 chunks
chunk_size= 500, overlap=0   → 2 chunks
chunk_size=1000, overlap=0   → 2 chunks


In [7]:
# Experiment 2: How does overlap affect chunks?
# With overlap, adjacent chunks share some text — notice the repeated content
chunks = chunk_pages(pages, chunk_size=100, chunk_overlap=50)
if len(chunks) >= 2:
    print(f"Chunk 0: ...{chunks[0]['text'][-60:]}")
    print(f"Chunk 1: {chunks[1]['text'][:60]}...")
    print(f"\nNotice the overlap — both chunks contain some of the same text.")
    print(f"This ensures context isn't lost at chunk boundaries.")

Chunk 0: ...5 million dollars.
Operating margins improved to 23 percent.
Chunk 1: Operating margins improved to 23 percent.
The company expand...

Notice the overlap — both chunks contain some of the same text.
This ensures context isn't lost at chunk boundaries.


In [8]:
# Experiment 3: What happens with an empty page?
pages_with_empty = [
    {"page_number": 1, "text": ""},
    {"page_number": 2, "text": "Only this page has content."},
]
chunks = chunk_pages(pages_with_empty, chunk_size=1000, chunk_overlap=0)
print(f"Chunks: {len(chunks)}")
print(f"All from page: {[c['page_number'] for c in chunks]}")
print("Empty pages are skipped — no empty chunks pollute the vector store.")

Chunks: 1
All from page: [2]
Empty pages are skipped — no empty chunks pollute the vector store.


## Check Your Understanding

1. **Why do we use `BytesIO` instead of reading the PDF file directly? What Go pattern is this similar to?**

2. **Why does chunk overlap matter for RAG? What would happen if overlap was 0?**

3. **The RecursiveCharacterTextSplitter tries separators in order: `\n\n`, `\n`, ` `, then characters. Why this order? What would happen if it only split on spaces?**